# LifeMate trên Google Colab: smoke train và QLoRA

Notebook này chuẩn bị dữ liệu **train có nhãn**, tải model nền, chạy smoke train và sao lưu adapter lên Google Drive. Chưa có model nào được train trong bundle. Đây là workflow phát triển; hồ sơ nộp cuối theo hướng dẫn hiện có là Docker package, không phải notebook.

Bundle chỉ chứa `train.jsonl` và tools catalog cần cho SFT. Không đưa thư mục `*_environment/`, `public_test`, `private_test` hay model weights lên Colab. Vì thiếu môi trường ToolEnv, notebook không chạy rollout chấm state/tool-call; hãy đánh giá adapter trên môi trường được BTC cho phép bằng `zaic_tools`.


## Trước khi bắt đầu

1. Tạo notebook trong Google Colab và chọn **Runtime → Change runtime type → GPU**.
2. Upload `lifemate-colab-bundle-*.tar.gz` vào thư mục gốc `MyDrive`.
3. Đảm bảo đội được phép xử lý dữ liệu train trên Google Colab theo điều khoản BTC và chính sách dữ liệu của đội.
4. Bundle không có model weights; bước tải model cần internet. Khi train, Google Drive sẽ lưu adapter để không mất khi runtime bị reset.

Luồng: mount Drive → giải nén source/train → kiểm tra GPU và disk → cài thư viện → tải model → tạo SFT chỉ từ train → smoke train → tùy chọn full train → sao lưu adapter. Full train mặc định tắt.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")
bundles = sorted(
    DRIVE_ROOT.glob("lifemate-colab-bundle-*.tar.gz"),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)
if not bundles:
    raise FileNotFoundError("Upload lifemate-colab-bundle-*.tar.gz to the root of MyDrive, then rerun this cell.")
BUNDLE_PATH = bundles[0]
print("Using bundle:", BUNDLE_PATH.name)
RUN_ID = time.strftime("%Y%m%d-%H%M%S")
RUN_DIR = DRIVE_ROOT / "lifemate_runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Persistent run directory:", RUN_DIR)


## 1. Bung source và dữ liệu train an toàn

Bundle đã được lọc trước: chỉ có `train.jsonl` và `train_tools_catalog.json` từ split train. Cell này không mở hay bung thư mục môi trường.


In [ ]:
WORK_ROOT = Path("/content/lifemate_job")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run(["tar", "-xzf", str(BUNDLE_PATH), "-C", str(WORK_ROOT)], check=True)
PROJECT_DIR = WORK_ROOT / "lifemate"
if not (PROJECT_DIR / "scripts" / "download_models.py").exists():
    raise FileNotFoundError(f"Source bundle không hợp lệ: {PROJECT_DIR}")
os.chdir(PROJECT_DIR)
subprocess.run(["sha256sum", "-c", "COLAB_DATA_SHA256SUMS"], cwd=PROJECT_DIR, check=True)
TRAIN_DIR = PROJECT_DIR / "data/raw/train+public_test/train"
for required in (TRAIN_DIR / "train.jsonl", TRAIN_DIR / "train_tools_catalog.json"):
    if not required.is_file():
        raise FileNotFoundError(f"Bundle thiếu dữ liệu train cần thiết: {required}")
print("Project:", PROJECT_DIR)
print("Train labels and tools catalog are present; environment directories are not included.")


## 2. Kiểm tra GPU, bộ nhớ và dung lượng

Cấu hình sequence length ban đầu khá thận trọng để smoke train có cơ hội chạy trên GPU Colab phổ biến. Nếu gặp CUDA OOM, giảm xuống 1024 hoặc dùng GPU có VRAM lớn hơn.


In [ ]:
subprocess.run(["nvidia-smi"], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Colab chưa cấp CUDA GPU. Chọn Runtime → Change runtime type → GPU rồi chạy lại.")
props = torch.cuda.get_device_properties(0)
VRAM_GB = props.total_memory / (1024 ** 3)
FREE_GB = shutil.disk_usage("/content").free / (1024 ** 3)
MAX_SEQUENCE_LENGTH = 4096 if VRAM_GB >= 22 else 2048 if VRAM_GB >= 14 else 1024
TRAIN_BATCH_SIZE = 4 if VRAM_GB >= 38 else 2 if VRAM_GB >= 22 else 1
GRAD_ACCUM_STEPS = 4 if TRAIN_BATCH_SIZE >= 4 else 8 if TRAIN_BATCH_SIZE >= 2 else 16
USE_FLASH_ATTN = VRAM_GB >= 22
print(f"GPU: {props.name}; VRAM: {VRAM_GB:.1f} GB; disk trống: {FREE_GB:.1f} GB")
print(f"Tự động cấu hình: seq={MAX_SEQUENCE_LENGTH}, batch_size={TRAIN_BATCH_SIZE}, grad_accum={GRAD_ACCUM_STEPS}, flash_attn={USE_FLASH_ATTN}")
if FREE_GB < 15:
    raise RuntimeError("Disk còn dưới 15 GB; cần runtime có thêm dung lượng trước khi tải model.")


## 3. Cài thư viện

Colab cung cấp PyTorch CUDA. Requirements của repo không cài lại PyTorch; cell cài các thư viện train theo phiên bản đã pin. Sau khi cài xong, nếu Colab yêu cầu restart runtime, hãy restart rồi chạy lại từ đầu.


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements-train.txt"],
    cwd=PROJECT_DIR,
    check=True,
)
import torch
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda, "available:", torch.cuda.is_available())


## 4. Tải model và tạo SFT từ train

Qwen2.5-3B model weights tải từ Hugging Face vào runtime Colab (không tải ASR vì không cần cho train). `prepare_sft.py` chỉ đọc `train.jsonl` và catalog; nó chia train thành train/dev/holdout theo request để tránh rò rỉ giữa các nhóm câu giống nhau.


In [ ]:
subprocess.run([sys.executable, "scripts/download_models.py", "--models", "llm"], cwd=PROJECT_DIR, check=True)
subprocess.run(
    [sys.executable, "training/prepare_sft.py", "--data-root", "data/raw/train+public_test", "--augment-masking"],
    cwd=PROJECT_DIR,
    check=True,
)
manifest = json.loads((PROJECT_DIR / "data/processed/sft/manifest.json").read_text(encoding="utf-8"))
print("SFT split:", manifest["samples"])
print("SFT records:", manifest["records"])


## 5. Smoke train QLoRA

Chạy thử trên 64 mẫu train, 16 mẫu dev, 1 epoch. Mục tiêu là kiểm tra data pipeline, CUDA, optimizer và lưu adapter. Đây chưa phải kết quả thi; chỉ mở full train sau khi smoke hoàn tất và adapter được sao lưu.


In [ ]:
smoke_command = [
    sys.executable, "training/train_lora.py",
    "--train-jsonl", "data/processed/sft/train.jsonl",
    "--dev-jsonl", "data/processed/sft/dev.jsonl",
    "--limit-train-samples", "64",
    "--limit-dev-samples", "16",
    "--epochs", "1",
    "--batch-size", str(TRAIN_BATCH_SIZE),
    "--gradient-accumulation-steps", str(GRAD_ACCUM_STEPS),
    "--max-sequence-length", str(MAX_SEQUENCE_LENGTH),
    "--output-dir", "models/lifemate-qwen2.5-3b-smoke",
]
if USE_FLASH_ATTN:
    smoke_command.append("--use-flash-attention-2")
subprocess.run(smoke_command, cwd=PROJECT_DIR, check=True)
SMOKE_ADAPTER = PROJECT_DIR / "models/lifemate-qwen2.5-3b-smoke/final"
if not SMOKE_ADAPTER.is_dir():
    raise FileNotFoundError(f"Không thấy adapter smoke: {SMOKE_ADAPTER}")
SMOKE_BACKUP = RUN_DIR / "lifemate-qwen2.5-3b-smoke-adapter"
shutil.copytree(SMOKE_ADAPTER, SMOKE_BACKUP, dirs_exist_ok=True)
print("Smoke adapter saved to Drive:", SMOKE_BACKUP)


## 6. Tùy chọn full train

Full train mặc định tắt để tránh chạy nhầm một job dài. Sau khi smoke train thành công và adapter đã được sao lưu, đổi `RUN_FULL_TRAINING` thành `True` rồi chạy cell này. Thời gian phụ thuộc GPU Colab và có thể vượt thời lượng một phiên; dùng checkpoint/adapter được sao lưu trên Drive.


In [ ]:
RUN_FULL_TRAINING = False
FULL_OUTPUT_DIR = RUN_DIR / "lifemate-qwen2.5-3b-full-training"
FULL_ADAPTER = FULL_OUTPUT_DIR / "final"
if RUN_FULL_TRAINING:
    full_command = [
        sys.executable, "training/train_lora.py",
        "--train-jsonl", "data/processed/sft/train.jsonl",
        "--dev-jsonl", "data/processed/sft/dev.jsonl",
        "--epochs", "3",
        "--batch-size", str(TRAIN_BATCH_SIZE),
        "--gradient-accumulation-steps", str(GRAD_ACCUM_STEPS),
        "--lora-r", "32",
        "--lora-alpha", "64",
        "--learning-rate", "2e-4",
        "--max-sequence-length", str(MAX_SEQUENCE_LENGTH),
        "--output-dir", str(FULL_OUTPUT_DIR),
    ]
    if USE_FLASH_ATTN:
        full_command.append("--use-flash-attention-2")
    subprocess.run(full_command, cwd=PROJECT_DIR, check=True)
    if not FULL_ADAPTER.is_dir():
        raise FileNotFoundError(f"Không thấy full adapter: {FULL_ADAPTER}")
    print("Full adapter:", FULL_ADAPTER)
    print("Trainer checkpoints are saved under:", FULL_OUTPUT_DIR)
else:
    print("Full train đang tắt. Muốn chạy thì đổi RUN_FULL_TRAINING = True và chạy lại cell này.")


## 7. Sao lưu adapter lên Google Drive

Smoke adapter được sao lưu lên Drive ngay sau khi train xong. Full train ghi checkpoints trực tiếp vào Drive; chạy cell metadata sau full train để cập nhật run info. Model weights nền không được sao lưu; khi đánh giá trên máy khác, tải đúng revision model theo `configs/baseline.json`.


In [ ]:
(RUN_DIR / "run_info.json").write_text(
    json.dumps({
        "run_id": RUN_ID,
        "gpu": props.name,
        "vram_gb": round(VRAM_GB, 2),
        "max_sequence_length": MAX_SEQUENCE_LENGTH,
        "full_training": RUN_FULL_TRAINING,
        "seed": 2026,
        "smoke_adapter": str(SMOKE_BACKUP),
        "full_adapter": str(FULL_ADAPTER) if RUN_FULL_TRAINING else None,
    }, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print("Smoke adapter and run info are saved to:", RUN_DIR)
if RUN_FULL_TRAINING:
    print("Full adapter is saved to:", FULL_ADAPTER)


## 8. Trực quan hóa đường cong Loss (Train & Validation)

Vẽ đồ thị Loss từ `trainer_state.json` để kiểm tra độ hội tụ và phát hiện sớm hiện tượng Overfitting/Underfitting.


In [ ]:
import json
import matplotlib.pyplot as plt

target_dir = FULL_OUTPUT_DIR if RUN_FULL_TRAINING else (PROJECT_DIR / "models/lifemate-qwen2.5-3b-smoke")
state_file = target_dir / "trainer_state.json"
if state_file.is_file():
    state = json.loads(state_file.read_text(encoding="utf-8"))
    log_history = state.get("log_history", [])
    train_steps, train_losses = [], []
    eval_steps, eval_losses = [], []
    for entry in log_history:
        if "loss" in entry and "step" in entry:
            train_steps.append(entry["step"])
            train_losses.append(entry["loss"])
        if "eval_loss" in entry and "step" in entry:
            eval_steps.append(entry["step"])
            eval_losses.append(entry["eval_loss"])
    
    plt.figure(figsize=(10, 5))
    if train_losses:
        plt.plot(train_steps, train_losses, label="Training Loss (per step)", color="royalblue", alpha=0.8)
    if eval_losses:
        plt.plot(eval_steps, eval_losses, label="Validation Loss (per epoch)", color="crimson", marker="o", linewidth=2)
    plt.xlabel("Optimization Steps")
    plt.ylabel("Cross Entropy Loss")
    plt.title(f"LifeMate Qwen2.5-3B Training Convergence ({props.name})")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.5)
    loss_plot_path = RUN_DIR / "loss_curve.png"
    plt.savefig(loss_plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    print("Đã lưu biểu đồ Loss vào:", loss_plot_path)
else:
    print("Chưa thấy trainer_state.json. Chạy smoke hoặc full train trước.")


## Gỡ lỗi nhanh và bước tiếp theo

- **CUDA False:** bật GPU trong Runtime settings rồi reconnect.
- **CUDA out of memory:** đặt `MAX_SEQUENCE_LENGTH = 1024`, restart runtime và chạy lại từ đầu.
- **Thiếu disk:** chọn runtime còn ít nhất 15 GB trống trước khi tải Qwen và PhoWhisper.
- **Runtime sắp ngắt:** đợi smoke train hoàn tất rồi chạy cell sao lưu; đừng đợi tới khi phiên Colab tự dừng.
- **Đánh giá chất lượng:** chuyển adapter về môi trường được phép có `zaic_tools`, rồi chạy đánh giá dev qua ToolEnv. Notebook này không chứa `*_environment/` và không chấm state/tool-call.
- **Nộp BTC:** notebook và adapter riêng không thay Docker solution; đối chiếu lại hướng dẫn nộp chính thức trước khi đóng gói cuối.
